In [2]:
"""Reviewer 3 pipeline for 5, 10, and 50 ug/L datasets (XGBoost replaces LinearRegression).

Input CSV is transposed:
  row 1: Y, followed by sample labels (0/1)
  rows 2+: feature name, followed by normalized values for each sample

This script uses a stratified random 80/20 holdout, classic shuffled stratified CV
on the training set, fold-local preprocessing/feature selection, and benchmarks,
PFI, and SHAP for the final model. See notes in the accompanying answer.
"""
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, f1_score, matthews_corrcoef, precision_score, recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold, RandomizedSearchCV, ParameterGrid, train_test_split,
)
from sklearn.model_selection import cross_val_predict
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.utils import resample
import joblib

# ------------------------------ Configuration ------------------------------
DATA_DIR = Path(r"E:\NO-NC\normalization\revise")
INPUT_FILES = {
    5: DATA_DIR / "extract_20261019_train_005.csv",
    10: DATA_DIR / "extract_20261019_train_010.csv",
    50: DATA_DIR / "extract_20261019_train_050.csv",
}
RESULTS_DIR = DATA_DIR / "reviewer3_threshold_results"
RANDOM_STATE = 20261019
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3
N_ITER_SEARCH = 20  # Randomly sample up to 20 parameter combinations per search.
COMPARISON_THRESHOLD = 0.5  # Initial/fallback classification threshold.
THRESHOLD_CANDIDATES = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
CORR_THRESHOLD = 0.85
PFI_TOP_K = 15
PFI_REPEATS = 20

# These are nominal categories in the current variable list. Their normalized
# codes are treated as categories (not as ordered continuous measurements).
CATEGORICAL = ["Parent material", "Climate classification", "Land use", "Texture"]
COORD_LON = "Longitude"
COORD_LAT = "Latitude"
# Explicitly forbid study-label-derived interpolation predictors if present.
LEAKY_NAMES = {
    "Binary value interpolations of 5 μg/L",
    "Binary value interpolations of 10 μg/L",
    "Binary value interpolations of 50 μg/L",
}

# ------------------------------- Load data ---------------------------------
def load_transposed_csv(path: Path):
    raw = pd.read_csv(path, header=None, encoding="utf-8-sig", dtype=str)
    raw = raw.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
    if str(raw.iat[0, 0]).strip().lower() not in {"y", "target", "learning target"}:
        raise ValueError(f"Expected first row label Y; found {raw.iat[0,0]!r}")

    y = pd.to_numeric(raw.iloc[0, 1:], errors="coerce").reset_index(drop=True)
    feature_names = raw.iloc[1:, 0].astype(str).str.strip().tolist()
    if len(feature_names) != len(set(feature_names)):
        raise ValueError("Duplicate feature names found in first column")
    if any(name in LEAKY_NAMES for name in feature_names):
        bad = [name for name in feature_names if name in LEAKY_NAMES]
        raise ValueError(f"Target-derived interpolation predictors must be removed: {bad}")

    # Transpose so rows are samples and columns are predictors.
    X = raw.iloc[1:, 1:].T.reset_index(drop=True)
    X.columns = feature_names
    X = X.apply(pd.to_numeric, errors="coerce")
    # The file uses -9999 as a NoData sentinel; convert it before imputation.
    X = X.replace([-9999, -9999.0, np.inf, -np.inf], np.nan)
    keep = y.notna()
    X, y = X.loc[keep].reset_index(drop=True), y.loc[keep].astype(int).reset_index(drop=True)
    if not set(y.unique()).issubset({0, 1}):
        raise ValueError(f"Y must contain only 0/1; got {sorted(y.unique())}")

    # Do not remove all-missing or constant predictors here: doing so would use
    # information from the eventual validation/test rows before splitting.
    # These columns are screened using the training portion of each fold below.
    return X, y, []


def usable_training_features(X_train):
    """Return nonempty, nonconstant predictors using training data only."""
    return [c for c in X_train.columns
            if not X_train[c].isna().all() and X_train[c].nunique(dropna=True) > 1]


def mixed_mutual_info_scores(X, y):
    """MI scores for dense preprocessed data, treating binary columns as discrete."""
    values = np.asarray(X, dtype=float)
    discrete = np.zeros(values.shape[1], dtype=bool)
    for j in range(values.shape[1]):
        col = values[:, j]
        col = col[np.isfinite(col)]
        discrete[j] = np.unique(col).size <= 2
    return mutual_info_classif(
        values, y, discrete_features=discrete, random_state=RANDOM_STATE
    )


def fitted_model_feature_names(fitted_pipeline):
    """Get transformed feature names with compatibility for older sklearn releases."""
    prep = fitted_pipeline.named_steps["prep"]
    names = []
    # ColumnTransformer.get_feature_names_out() requires every nested step to
    # implement that API. Older sklearn releases lack it on SimpleImputer, so
    # derive the output schema from the fitted transformer components instead.
    for name, transformer, columns in prep.transformers_:
        if name == "remainder" or transformer == "drop":
            continue
        columns = list(columns) if not isinstance(columns, str) else [columns]
        # ColumnTransformer skips fitting a transformer assigned zero columns.
        # PFI can remove every nominal variable in a fold, leaving its OneHotEncoder unfitted.
        if not columns:
            continue
        if transformer == "passthrough":
            names.extend(columns)
        elif name == "num":
            # Numeric imputation and optional scaling preserve one column per input.
            names.extend(columns)
        elif name == "cat":
            encoder = transformer.named_steps["onehot"]
            if hasattr(encoder, "get_feature_names_out"):
                names.extend(encoder.get_feature_names_out(columns).tolist())
            elif hasattr(encoder, "get_feature_names"):
                names.extend(encoder.get_feature_names(columns).tolist())
            else:
                # Last-resort compatibility for very old sklearn versions.
                for feature, categories in zip(columns, encoder.categories_):
                    names.extend(f"{feature}_{category}" for category in categories)
        else:
            raise RuntimeError(f"Cannot derive output feature names for transformer {name!r}")
    if "select" in fitted_pipeline.named_steps:
        selector = fitted_pipeline.named_steps["select"]
        names = [name for name, keep in zip(names, selector.get_support()) if keep]
    model_width = getattr(fitted_pipeline.named_steps["model"], "n_features_in_", None)
    if model_width is not None and len(names) != model_width:
        raise RuntimeError(
            f"Feature-name count ({len(names)}) does not match model input width "
            f"({model_width})"
        )
    return names


def stratified_split(X, y):
    """Random 80/20 holdout, preserving the 0/1 proportions in expectation."""
    idx = np.arange(len(y))
    train_idx, test_idx = train_test_split(
        idx, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )
    return train_idx, test_idx


def correlation_filter(X_train, numeric_cols, threshold=CORR_THRESHOLD):
    """Greedy Spearman filter on continuous variables; keep nominal variables."""
    corr_cols = [c for c in numeric_cols if X_train[c].nunique(dropna=True) > 2]
    if len(corr_cols) < 2:
        return list(X_train.columns)
    corr = X_train[corr_cols].corr(method="spearman").abs().fillna(0)
    kept = []
    for col in corr_cols:
        if all(corr.loc[col, prior] < threshold for prior in kept):
            kept.append(col)
    nominal = [c for c in X_train.columns if c not in corr_cols]
    return nominal + kept


def pfi_select(X_train, y_train, categorical_cols, numeric_cols):
    """Select predictors by CatBoost-regressor PFI on an inner stratified holdout."""
    try:
        from catboost import CatBoostRegressor
    except ImportError as exc:
        raise RuntimeError(
            "CatBoost is required for PFI feature selection. Install it in the "
            "active Python environment with: pip install catboost"
        ) from exc

    local_idx = np.arange(len(y_train))
    tr_idx, va_idx = train_test_split(
        local_idx, test_size=0.25, random_state=RANDOM_STATE, stratify=y_train
    )
    prep = make_preprocessor(categorical_cols, numeric_cols, scale_numeric=False)
    catboost_model = CatBoostRegressor(
        iterations=400, depth=6, learning_rate=0.05,
        loss_function="RMSE", random_seed=RANDOM_STATE,
        verbose=False, thread_count=1, allow_writing_files=False,
    )
    pipe = Pipeline([("prep", prep), ("model", catboost_model)])
    pipe.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
    pfi = permutation_importance(
        pipe, X_train.iloc[va_idx], y_train.iloc[va_idx],
        scoring=average_precision_scorer, n_repeats=PFI_REPEATS,
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    names = X_train.columns.to_list()
    ranking = pd.DataFrame({"feature": names, "pfi_mean": pfi.importances_mean,
                            "pfi_sd": pfi.importances_std}).sort_values("pfi_mean", ascending=False)
    # Retain only features with positive mean contribution, up to the configured maximum.
    positive = ranking.loc[ranking["pfi_mean"] > 0]
    selected = positive.head(min(PFI_TOP_K, len(positive)))["feature"].tolist()
    if not selected:
        selected = ranking.head(min(PFI_TOP_K, len(ranking)))["feature"].tolist()
    return selected, ranking


# ------------------------------ Models --------------------------------------
def make_preprocessor(cat_cols, num_cols, scale_numeric):
    cat_cols = [c for c in cat_cols if c in num_cols or c in X_COLUMNS]
    numeric = [c for c in num_cols if c not in cat_cols]
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scale", StandardScaler()))
    # Compatibility: sklearn <=1.1 uses sparse=False; newer versions use
    # sparse_output=False. Both create a dense matrix for downstream selectors.
    import inspect
    if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    else:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    cat_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", ohe),
    ])
    return ColumnTransformer([
        ("num", Pipeline(num_steps), numeric),
        ("cat", cat_pipe, cat_cols),
    ], remainder="drop", verbose_feature_names_out=False)


X_COLUMNS = []
def model_specs():
    # Regressors are fitted to the binary 0/1 target. Their continuous
    # predictions are treated as risk scores; threshold selection is done on OOF scores.
    # LogisticRegression is retained as the interpretable baseline requested
    # by the reviewer and contributes predict_proba scores.
    models = {
        "LogisticRegression": (
            LogisticRegression(max_iter=3000, solver="liblinear", random_state=RANDOM_STATE),
            {"model__C": [0.01, 0.1, 1.0, 10.0]}, True),
        "RandomForestRegressor": (
            RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=1),
            {"model__n_estimators": [200, 400, 600, 800],
             "model__max_depth": [None, 10, 20, 30],
             "model__min_samples_leaf": [1, 2, 3, 5]}, False),
        "SVR": (
            SVR(), {"model__C": [0.1, 1, 10, 100],
                    "model__epsilon": [0.01, 0.05, 0.1, 0.2]}, True),
        "MLPRegressor": (
            MLPRegressor(max_iter=500, early_stopping=True, random_state=RANDOM_STATE),
            {"model__hidden_layer_sizes": [(32,), (64,), (64, 32), (128, 64)],
             "model__alpha": [0.00001, 0.0001, 0.001, 0.01]}, True),
    }
    try:
        from xgboost import XGBRegressor
    except ImportError as exc:
        raise RuntimeError(
            "XGBoost is required for this run. Install it in the active Python "
            "environment with: pip install xgboost"
        ) from exc
    models["XGBoostRegressor"] = (XGBRegressor(
        objective="reg:squarederror", eval_metric="rmse",
        random_state=RANDOM_STATE, n_jobs=1, tree_method="hist",
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0),
        {"model__n_estimators": [200, 400, 600, 800],
         "model__max_depth": [2, 3, 4, 6],
         "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
         "model__subsample": [0.6, 0.7, 0.8, 1.0],
         "model__colsample_bytree": [0.6, 0.7, 0.8, 1.0],
         "model__reg_lambda": [0.1, 1.0, 5.0, 10.0]}, False)
    try:
        from lightgbm import LGBMRegressor
        models["LightGBMRegressor"] = (LGBMRegressor(
            objective="regression", random_state=RANDOM_STATE, n_jobs=1, verbosity=-1),
            {"model__n_estimators": [200, 400, 600, 800],
             "model__num_leaves": [7, 15, 31, 63],
             "model__learning_rate": [0.01, 0.03, 0.05, 0.1]}, False)
    except ImportError: warnings.warn("lightgbm not installed; skipping LightGBMRegressor")
    try:
        from catboost import CatBoostRegressor
        models["CatBoostRegressor"] = (CatBoostRegressor(
            verbose=False, random_seed=RANDOM_STATE, thread_count=1,
            loss_function="RMSE", allow_writing_files=False),
            {"model__iterations": [300, 500, 800, 1000],
             "model__depth": [4, 5, 6, 8],
             "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
             "model__l2_leaf_reg": [1, 3, 5, 8]}, False)
    except ImportError: warnings.warn("catboost not installed; skipping CatBoostRegressor")
    return models


def continuous_score(estimator, X_values):
    """Get a continuous score from a classifier or regressor."""
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(X_values)[:, 1]
    return np.asarray(estimator.predict(X_values), dtype=float).reshape(-1)


def average_precision_scorer(estimator, X_values, y_true):
    # AP accepts continuous scores, including regressor predictions.
    return average_precision_score(y_true, continuous_score(estimator, X_values))

def expected_calibration_error(y_true, probability, n_bins=10):
    """ECE with 10 equal-width bins covering [0, 1], including score 1 in the last bin."""
    y_true = np.asarray(y_true, dtype=int)
    probability = np.clip(np.asarray(probability, dtype=float), 0.0, 1.0)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_id = np.minimum(np.digitize(probability, edges[1:-1], right=False), n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        mask = bin_id == b
        if np.any(mask):
            observed_rate = float(y_true[mask].mean())
            mean_probability = float(probability[mask].mean())
            ece += float(mask.mean()) * abs(observed_rate - mean_probability)
    return float(ece)


def calculate_metrics(y_true, score, threshold=COMPARISON_THRESHOLD, predicted_class=None):
    """Evaluate thresholded classes and probability-like scores."""
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float).reshape(-1)
    # Regression outputs are interpreted as risk scores and clipped to [0,1]
    # only when a probability-scale metric (ECE) is calculated.
    probability = np.clip(score, 0.0, 1.0)
    pred = (score > threshold).astype(int) if predicted_class is None else np.asarray(predicted_class, dtype=int)
    if len(pred) != len(y_true):
        raise ValueError("predicted_class and y_true must have equal lengths")
    tn = int(np.sum((y_true == 0) & (pred == 0)))
    fp = int(np.sum((y_true == 0) & (pred == 1)))
    return {
        "classification_threshold": float(threshold),
        "TPR": recall_score(y_true, pred, zero_division=0),
        "FPR": fp / (fp + tn) if (fp + tn) else 0.0,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "MCC": matthews_corrcoef(y_true, pred),
        "ROC_AUC": roc_auc_score(y_true, score),
        "PR_AUC": average_precision_score(y_true, score),
        "ECE_10_equal_width_bins": expected_calibration_error(y_true, probability, n_bins=10),
    }

def run(CSV_PATH, OUT_DIR, target_threshold):
    global X_COLUMNS
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    X, y, dropped = load_transposed_csv(CSV_PATH)
    X_COLUMNS = X.columns.tolist()
    warnings.warn("Predictors in this CSV are already normalized. For strict fold-local preprocessing, rerun from raw predictor data if available.")
    print(f"\n===== Target threshold: {target_threshold} ug/L =====")
    print(f"Full data: n={len(y)}, p={X.shape[1]}, class counts={y.value_counts().sort_index().to_dict()}")
    print("Dropped before split:", dropped)
    print("NoData (-9999) cells converted to missing; counts:")
    print(X.isna().sum().loc[lambda s: s > 0].to_dict())

    # G_As_05/G_As_10/G_As_50 are retained as candidate external predictors.
    # Confirm their provenance is independent of this study's surface-water Y.
    external_proxy_cols = [c for c in X if c.lower() in {"g_as_05", "g_as_10", "g_as_50"}]
    print("Retained G_As predictors; verify independent external provenance:", external_proxy_cols)

    train_idx, test_idx = stratified_split(X, y)
    X_train = X.iloc[train_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    X_test = X.iloc[test_idx].reset_index(drop=True)
    y_test = y.iloc[test_idx].reset_index(drop=True)
    split_info = {
        "random_state": RANDOM_STATE,
        "split": "random stratified 80/20",
        "n_train": int(len(y_train)), "n_test": int(len(y_test)),
        "train_class_counts": {str(k): int(v) for k, v in y_train.value_counts().sort_index().items()},
        "test_class_counts": {str(k): int(v) for k, v in y_test.value_counts().sort_index().items()},
        "train_class_proportions": {str(k): float(v) for k, v in y_train.value_counts(normalize=True).sort_index().items()},
        "test_class_proportions": {str(k): float(v) for k, v in y_test.value_counts(normalize=True).sort_index().items()},
    }
    with open(OUT_DIR / "split_info.json", "w", encoding="utf-8") as f:
        json.dump(split_info, f, ensure_ascii=False, indent=2)
    print("Train class proportions:", split_info["train_class_proportions"])
    print("Test class proportions:", split_info["test_class_proportions"])

    outer_cv = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    model_bank = model_specs()
    print("Models included in training-CV comparison:", list(model_bank))
    if "XGBoostRegressor" not in model_bank:
        raise RuntimeError("XGBoostRegressor was not registered; stopping before model comparison.")
    feature_modes = ["all", "correlation", "SelectKBest", "PFI"]
    oof_rows, fold_rows = [], []
    summary_rows = []
    threshold_selection_rows = []

    for mode in feature_modes:
        for model_name, (estimator, param_grid, scale) in model_bank.items():
            oof_score = np.full(len(y_train), np.nan)
            oof_pred = np.full(len(y_train), -1, dtype=int)
            oof_threshold = np.full(len(y_train), np.nan, dtype=float)
            oof_fold = np.full(len(y_train), -1, dtype=int)
            fold_scores = []
            for fold, (tr, va) in enumerate(outer_cv.split(X_train, y_train), start=1):
                Xtr, Xva = X_train.iloc[tr].copy(), X_train.iloc[va].copy()
                ytr, yva = y_train.iloc[tr], y_train.iloc[va]
                selected = usable_training_features(Xtr)
                if not selected:
                    raise ValueError(f"No usable predictors in outer training fold {fold}")
                # Apply the training-fold schema to its validation fold.
                Xtr, Xva = Xtr[selected], Xva[selected]
                fold_cats = [c for c in CATEGORICAL if c in selected]
                fold_nums = [c for c in selected if c not in fold_cats]
                pfi_rank = None

                if mode == "correlation":
                    selected = correlation_filter(Xtr, fold_nums)
                elif mode == "PFI":
                    selected, pfi_rank = pfi_select(Xtr, ytr, fold_cats, fold_nums)
                    pfi_rank["fold"] = fold
                    pfi_rank["selection_model"] = "CatBoostRegressor"
                    pfi_rank.to_csv(OUT_DIR / f"PFI_{model_name}_fold{fold}.csv", index=False)

                used_cats = [c for c in fold_cats if c in selected]
                used_nums = [c for c in selected if c not in used_cats]
                prep = make_preprocessor(used_cats, used_nums, scale)
                steps = [("prep", prep)]
                if mode == "SelectKBest":
                    steps.append(("select", SelectKBest(score_func=mixed_mutual_info_scores)))
                steps.append(("model", clone(estimator)))
                pipe = Pipeline(steps)
                grid = dict(param_grid)
                if mode == "SelectKBest":
                    grid["select__k"] = [5, 10, 20, "all"]

                inner_cv = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True,
                                           random_state=RANDOM_STATE + fold)
                inner_splits = list(inner_cv.split(Xtr[selected], ytr))
                search = RandomizedSearchCV(
                    pipe, grid,
                    n_iter=min(N_ITER_SEARCH, len(list(ParameterGrid(grid)))),
                    scoring=average_precision_scorer, cv=inner_splits,
                    random_state=RANDOM_STATE + fold, n_jobs=-1,
                    refit=True, error_score="raise", verbose=0,
                )
                search.fit(Xtr[selected], ytr)

                # Choose a decision threshold from inner OOF predictions of this
                # outer-training fold only. The outer validation labels stay unseen.
                prediction_method = (
                    "predict_proba" if hasattr(search.best_estimator_, "predict_proba") else "predict"
                )
                inner_oof_raw = cross_val_predict(
                    clone(search.best_estimator_), Xtr[selected], ytr,
                    cv=inner_splits, method=prediction_method, n_jobs=-1,
                )
                if prediction_method == "predict_proba":
                    inner_oof_score = np.asarray(inner_oof_raw)[:, 1]
                else:
                    inner_oof_score = np.asarray(inner_oof_raw, dtype=float).reshape(-1)

                fold_threshold = COMPARISON_THRESHOLD
                threshold_rows = []
                candidates = sorted(set([COMPARISON_THRESHOLD] + THRESHOLD_CANDIDATES))
                for candidate in candidates:
                    candidate_pred = (inner_oof_score > candidate).astype(int)
                    threshold_rows.append({
                        "target_threshold_ug_L": target_threshold,
                        "feature_set": mode,
                        "model": model_name,
                        "outer_fold": fold,
                        "candidate_threshold": float(candidate),
                        "inner_OOF_F1": f1_score(ytr, candidate_pred, zero_division=0),
                        "inner_OOF_TPR": recall_score(ytr, candidate_pred, zero_division=0),
                        "inner_OOF_FPR": float(np.sum((ytr == 0) & (candidate_pred == 1)) / max(np.sum(ytr == 0), 1)),
                    })
                threshold_df = pd.DataFrame(threshold_rows)
                chosen_idx = threshold_df.sort_values(
                    ["inner_OOF_F1", "inner_OOF_TPR", "inner_OOF_FPR", "candidate_threshold"],
                    ascending=[False, False, True, False], kind="mergesort",
                ).index[0]
                fold_threshold = float(threshold_df.loc[chosen_idx, "candidate_threshold"])
                threshold_df["selected"] = threshold_df.index == chosen_idx
                threshold_selection_rows.extend(threshold_df.to_dict("records"))

                score = continuous_score(search.best_estimator_, Xva[selected])
                oof_score[va] = score
                oof_pred[va] = (score > fold_threshold).astype(int)
                oof_threshold[va] = fold_threshold
                oof_fold[va] = fold
                metrics = calculate_metrics(yva, score, threshold=fold_threshold)
                fold_scores.append(metrics)
                selected_model_features = fitted_model_feature_names(search.best_estimator_)
                fold_rows.append({"feature_set": mode, "model": model_name, "fold": fold,
                                  "n_validation": len(va), "n_positive": int(yva.sum()),
                                  "threshold_selection_method": "inner-training OOF maximum F1; ties: higher TPR, lower FPR",
                                  "input_predictors": ";".join(selected),
                                  "input_predictor_count": len(selected),
                                  "selected_features": ";".join(selected_model_features),
                                  "selected_model_feature_count": len(selected_model_features),
                                  "feature_selection_unit": (
                                      "one-hot encoded columns" if mode == "SelectKBest"
                                      else "original explanatory variables"
                                  ),
                                  "best_params": json.dumps(search.best_params_, default=str), **metrics})

            if (np.isnan(oof_score).any() or np.isnan(oof_threshold).any()
                    or (oof_pred < 0).any() or (oof_fold < 0).any()):
                raise RuntimeError(f"Some training samples lack OOF predictions: {mode}/{model_name}")
            for i, (truth, score, pred, threshold, fold) in enumerate(
                zip(y_train, oof_score, oof_pred, oof_threshold, oof_fold)
            ):
                oof_rows.append({"train_sample_index": i, "original_sample_index": int(train_idx[i]),
                                 "feature_set": mode, "model": model_name,
                                 "y_true": int(truth), "oof_score": float(score),
                                 "outer_fold": int(fold),
                                 "classification_threshold": float(threshold),
                                 "predicted_class": int(pred)})
            # Pool outer-fold predictions for unbiased training-only model comparison.
            fdf = pd.DataFrame(fold_scores)
            fold_thresholds = fdf["classification_threshold"].astype(float)
            final_combo_threshold = float(fold_thresholds.median())
            pooled = calculate_metrics(
                y_train, oof_score, threshold=final_combo_threshold, predicted_class=oof_pred
            )
            summary_rows.append({"feature_set": mode, "model": model_name,
                                 **{f"OOF_{k}": v for k, v in pooled.items()},
                                 "selected_threshold": final_combo_threshold,
                                 "mean_outer_fold_threshold": float(fold_thresholds.mean()),
                                 "median_outer_fold_threshold": final_combo_threshold,
                                 "sd_outer_fold_threshold": float(fold_thresholds.std(ddof=1)),
                                 "threshold_selection_metric": "inner-training OOF maximum F1; ties: higher TPR, lower FPR",
                                 "fold_PR_AUC_mean": fdf["PR_AUC"].mean(),
                                 "fold_PR_AUC_sd": fdf["PR_AUC"].std(ddof=1)})
            print(
                f"done {mode} / {model_name}: train OOF "
                f"PR-AUC={pooled['PR_AUC']:.4f}, "
                f"classification cutoff={final_combo_threshold:.2f}, "
                f"MCC={pooled['MCC']:.4f}, ECE={pooled['ECE_10_equal_width_bins']:.4f}"
            )

    # For every model/feature strategy, report the most frequent fold-optimal
    # parameter set; all fold-specific settings remain in train_cv_fold_metrics.csv.
    settings_rows = []
    fold_param_df = pd.DataFrame(fold_rows)
    for (feature_set, model), group in fold_param_df.groupby(["feature_set", "model"], sort=False):
        counts = group["best_params"].value_counts(dropna=False)
        consensus = str(counts.index[0]) if len(counts) else "{}"
        settings_rows.append({
            "target_threshold_ug_L": target_threshold,
            "feature_set": feature_set,
            "model": model,
            "consensus_fold_optimal_params": consensus,
            "consensus_frequency": int(counts.iloc[0]) if len(counts) else 0,
            "outer_folds": int(len(group)),
        })
    pd.DataFrame(threshold_selection_rows).to_csv(OUT_DIR / "threshold_selection_metrics.csv", index=False)
    pd.DataFrame(fold_rows).to_csv(OUT_DIR / "train_cv_fold_metrics.csv", index=False)
    pd.DataFrame(oof_rows).to_csv(OUT_DIR / "train_oof_predictions.csv", index=False)
    pd.DataFrame([r for r in fold_rows if r["model"] == "XGBoostRegressor"]).to_csv(
        OUT_DIR / "xgboost_fold_metrics.csv", index=False
    )
    pd.DataFrame([r for r in oof_rows if r["model"] == "XGBoostRegressor"]).to_csv(
        OUT_DIR / "xgboost_oof_predictions.csv", index=False
    )
    summary = pd.DataFrame(summary_rows)
    # PR-AUC is primary for the imbalanced risk outcome; remaining requested
    # metrics provide transparent, deterministic tie-breakers. ECE and FPR are lower-is-better.
    summary = summary.sort_values(
        ["OOF_PR_AUC", "OOF_ROC_AUC", "OOF_Balanced_accuracy", "OOF_MCC",
         "OOF_TPR", "OOF_FPR", "OOF_ECE_10_equal_width_bins", "OOF_Accuracy", "OOF_Precision"],
        ascending=[False, False, False, False, False, True, True, False, False],
        kind="mergesort",
    ).reset_index(drop=True)
    summary.insert(0, "target_threshold_ug_L", target_threshold)
    # One recommended feature strategy and consensus tuning result per model.
    settings_df = pd.DataFrame(settings_rows)
    best_per_model = summary.drop_duplicates("model", keep="first").copy()
    best_per_model = best_per_model.merge(
        settings_df,
        on=["feature_set", "model", "target_threshold_ug_L"],
        how="left",
        validate="one_to_one",
    )
    best_per_model.to_csv(OUT_DIR / "optimal_settings_by_model.csv", index=False)
    pd.DataFrame([{
        "target_threshold_ug_L": target_threshold,
        "initial_threshold": COMPARISON_THRESHOLD,
        "candidate_thresholds": ", ".join(map(str, THRESHOLD_CANDIDATES)),
        "classification_rule": "predicted value > selected threshold is high risk",
        "threshold_selection": "inner-training OOF maximum F1; ties prefer higher TPR then lower FPR",
        "probability_calibration_metric": "ECE with 10 equal-width bins over [0, 1]",
        "model_selection_primary_metric": "training OOF PR-AUC",
        "tie_break_order": "ROC-AUC, balanced accuracy, MCC, TPR, lower FPR, lower ECE, accuracy, precision",
    }]).to_csv(OUT_DIR / "evaluation_and_selection_rules.csv", index=False)
    if "XGBoostRegressor" not in set(summary["model"]):
        raise RuntimeError(
            "XGBoostRegressor is missing from the completed CV results; "
            "the comparison file was not accepted as complete."
        )
    summary.to_csv(OUT_DIR / "train_cv_model_feature_comparison.csv", index=False)
    summary.loc[summary["model"] == "XGBoostRegressor"].to_csv(
        OUT_DIR / "xgboost_cv_results.csv", index=False
    )
    best_model_row = summary.head(1).merge(
        best_per_model[["feature_set", "model", "consensus_fold_optimal_params", "consensus_frequency"]],
        on=["feature_set", "model"], how="left", validate="one_to_one"
    )
    best_model_row.to_csv(OUT_DIR / "best_model_by_threshold.csv", index=False)

    # Export the same three result tables for every model, including XGBoost,
    # so model-specific outputs have identical schemas and can be compared.
    per_model_dir = OUT_DIR / "per_model_results"
    per_model_dir.mkdir(parents=True, exist_ok=True)
    fold_metrics_df = pd.DataFrame(fold_rows)
    oof_predictions_df = pd.DataFrame(oof_rows)
    for name in sorted(summary["model"].unique()):
        name_summary = summary.loc[summary["model"] == name]
        name_folds = fold_metrics_df.loc[fold_metrics_df["model"] == name]
        name_oof = oof_predictions_df.loc[oof_predictions_df["model"] == name]
        name_summary.to_csv(per_model_dir / f"{name}_cv_summary.csv", index=False)
        name_folds.to_csv(per_model_dir / f"{name}_fold_metrics.csv", index=False)
        name_oof.to_csv(per_model_dir / f"{name}_oof_predictions.csv", index=False)

    print("Saved training-CV comparison outputs to", OUT_DIR)
    print("Saved standardized per-model summaries, fold metrics, and OOF predictions to", per_model_dir)
    print(summary.head(10).to_string(index=False))

    # Select the model/feature strategy using training-only CV; touch the test set
    # once after all choices are locked.
    best = summary.iloc[0]
    mode, model_name = best["feature_set"], best["model"]
    selected_threshold = float(best["selected_threshold"])
    print(f"CV-selected model/feature strategy: {model_name}/{mode}; median selected cutoff: > {selected_threshold:.2f}")

    estimator, param_grid, scale = model_bank[model_name]
    selected = usable_training_features(X_train)
    if not selected:
        raise ValueError("No usable predictors in the full training set")
    final_cats = [c for c in CATEGORICAL if c in selected]
    final_nums = [c for c in selected if c not in final_cats]
    if mode == "correlation":
        selected = correlation_filter(X_train[selected], final_nums)
    elif mode == "PFI":
        selected, final_pfi = pfi_select(X_train[selected], y_train, final_cats, final_nums)
        final_pfi.to_csv(OUT_DIR / "PFI_final_training_set.csv", index=False)
    used_cats = [c for c in final_cats if c in selected]
    used_nums = [c for c in selected if c not in used_cats]
    steps = [("prep", make_preprocessor(used_cats, used_nums, scale))]
    if mode == "SelectKBest":
        steps.append(("select", SelectKBest(score_func=mixed_mutual_info_scores)))
    steps.append(("model", clone(estimator)))
    final_pipe = Pipeline(steps)
    final_grid = dict(param_grid)
    if mode == "SelectKBest":
        final_grid["select__k"] = [5, 10, 20, "all"]
    final_cv = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    final_search = RandomizedSearchCV(
        final_pipe, final_grid,
        n_iter=min(N_ITER_SEARCH, len(list(ParameterGrid(final_grid)))),
        scoring=average_precision_scorer,
        cv=list(final_cv.split(X_train[selected], y_train)),
        random_state=RANDOM_STATE, n_jobs=-1,
        refit=True, error_score="raise",
    )
    final_search.fit(X_train[selected], y_train)

    test_score = continuous_score(final_search.best_estimator_, X_test[selected])
    test_metrics = calculate_metrics(y_test, test_score, threshold=selected_threshold)
    pd.DataFrame([{"target_threshold_ug_L": target_threshold, "feature_set": mode, "model": model_name,
                   "n_test": len(y_test), "n_positive": int(y_test.sum()), **test_metrics}]
                 ).to_csv(OUT_DIR / "independent_test_metrics.csv", index=False)
    pd.DataFrame({"original_sample_index": test_idx, "y_true": y_test,
                  "prediction_score": test_score,
                  "predicted_class": (test_score > selected_threshold).astype(int),
                  "classification_threshold": selected_threshold}
                 ).to_csv(OUT_DIR / "independent_test_predictions.csv", index=False)
    print("Independent test metrics:", test_metrics)

    joblib.dump({"pipeline": final_search.best_estimator_, "features": selected,
                 "feature_set": mode, "model": model_name,
                 "classification_threshold": selected_threshold}, OUT_DIR / "final_model.joblib")
    with open(OUT_DIR / "final_model_info.json", "w", encoding="utf-8") as f:
        json.dump({"model": model_name, "feature_set": mode,
                   "classification_threshold": selected_threshold,
                   "threshold_selection_metric": "inner-training OOF maximum F1; ties: higher TPR, lower FPR",
                   "selected_features": selected,
                   "best_params": final_search.best_params_, "class_counts_train": y_train.value_counts().to_dict(),
                   "class_counts_test": y_test.value_counts().to_dict()},
                  f, ensure_ascii=False, indent=2, default=str)

    # SHAP explains the final training-fit model; performance claims come from
    # training CV and the one-time independent test result.
    try:
        import shap
        fitted = final_search.best_estimator_
        Xt = fitted.named_steps["prep"].transform(X_train[selected])
        names = list(fitted.named_steps["prep"].get_feature_names_out())
        if "select" in fitted.named_steps:
            selector = fitted.named_steps["select"]
            Xt = selector.transform(Xt)
            names = [n for n, keep in zip(names, selector.get_support()) if keep]
        model = fitted.named_steps["model"]
        rng = np.random.default_rng(RANDOM_STATE)
        take = rng.choice(len(y_train), size=min(300, len(y_train)), replace=False)
        background = Xt[take[:min(100, len(take))]]
        explain_x = Xt[take]
        explainer = shap.Explainer(model, background, feature_names=names)
        sv = explainer(explain_x)
        values = sv.values
        if values.ndim == 3:
            values = values[:, :, 1]
        imp = pd.DataFrame({"feature": names, "mean_abs_shap": np.abs(values).mean(axis=0)})
        imp.sort_values("mean_abs_shap", ascending=False).to_csv(OUT_DIR / "shap_importance.csv", index=False)
        shap.summary_plot(values, explain_x, feature_names=names, show=False)
        import matplotlib.pyplot as plt
        plt.tight_layout(); plt.savefig(OUT_DIR / "shap_summary.png", dpi=300); plt.close()
    except Exception as exc:
        warnings.warn(f"SHAP step skipped/failed: {exc}")

    print("Model and independent test evaluation saved to", OUT_DIR)
    print("Risk raster and map-level uncertainty require the full predictor rasters;")
    print("this CSV supports sample-level CV, holdout testing, and interpretation only.")


if __name__ == "__main__":
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    all_optimal_settings = []
    all_best_models = []
    for target_threshold, csv_path in INPUT_FILES.items():
        if not csv_path.exists():
            raise FileNotFoundError(f"Input CSV not found: {csv_path}")
        output_dir = RESULTS_DIR / f"threshold_{target_threshold}"
        run(csv_path, output_dir, target_threshold)
        all_optimal_settings.append(pd.read_csv(output_dir / "optimal_settings_by_model.csv"))
        all_best_models.append(pd.read_csv(output_dir / "best_model_by_threshold.csv"))

    pd.concat(all_optimal_settings, ignore_index=True).to_csv(
        RESULTS_DIR / "optimal_settings_all_thresholds.csv", index=False
    )
    pd.concat(all_best_models, ignore_index=True).to_csv(
        RESULTS_DIR / "best_models_all_thresholds.csv", index=False
    )
    print("Consolidated optimal settings and best models saved to", RESULTS_DIR)



===== Target threshold: 5 ug/L =====
Full data: n=4038, p=31, class counts={0: 3450, 1: 588}
Dropped before split: []
NoData (-9999) cells converted to missing; counts:
{'Gravitational field': 1, 'TWI': 1, 'CEC': 1, 'Parent material': 1, 'Climate classification': 2, 'Land use': 1, 'G_As_05': 1, 'G_As_10': 1, 'G_As_50': 1, 'pH': 1, 'Precipitation': 2, 'Solar radiation': 2, 'Average temperature': 2, 'Texture': 1, 'Maximum temperature': 2, 'Minimum temperature': 2, 'Vapor pressure': 2, 'Wind speed': 2}
Retained G_As predictors; verify independent external provenance: ['G_As_05', 'G_As_10', 'G_As_50']
Train class proportions: {'0': 0.8544891640866873, '1': 0.14551083591331268}
Test class proportions: {'0': 0.8539603960396039, '1': 0.14603960396039603}
Models included in training-CV comparison: ['LogisticRegression', 'RandomForestRegressor', 'SVR', 'MLPRegressor', 'XGBoostRegressor', 'LightGBMRegressor', 'CatBoostRegressor']
done all / LogisticRegression: train OOF PR-AUC=0.3017, classific

Independent test metrics: {'classification_threshold': 0.2, 'TPR': 0.6440677966101694, 'FPR': 0.17101449275362318, 'Accuracy': 0.801980198019802, 'Precision': 0.3917525773195876, 'Balanced_accuracy': 0.7365266519282732, 'MCC': 0.3911024877743321, 'ROC_AUC': 0.7995701301891427, 'PR_AUC': 0.44529307704737686, 'ECE_10_equal_width_bins': 0.030060444521988618}
Model and independent test evaluation saved to E:\NO-NC\normalization\revise\reviewer3_threshold_results\threshold_5
Risk raster and map-level uncertainty require the full predictor rasters;
this CSV supports sample-level CV, holdout testing, and interpretation only.

===== Target threshold: 10 ug/L =====
Full data: n=4038, p=31, class counts={0: 3780, 1: 258}
Dropped before split: []
NoData (-9999) cells converted to missing; counts:
{'Gravitational field': 1, 'TWI': 1, 'CEC': 1, 'Parent material': 1, 'Climate classification': 2, 'Land use': 1, 'G_As_05': 1, 'G_As_10': 1, 'G_As_50': 1, 'pH': 1, 'Precipitation': 2, 'Solar radiation': 

Independent test metrics: {'classification_threshold': 0.15, 'TPR': 0.3269230769230769, 'FPR': 0.047619047619047616, 'Accuracy': 0.9121287128712872, 'Precision': 0.32075471698113206, 'Balanced_accuracy': 0.6396520146520146, 'MCC': 0.2768396936188341, 'ROC_AUC': 0.7422415547415548, 'PR_AUC': 0.23600102657919686, 'ECE_10_equal_width_bins': 0.009679071808067879}
Model and independent test evaluation saved to E:\NO-NC\normalization\revise\reviewer3_threshold_results\threshold_10
Risk raster and map-level uncertainty require the full predictor rasters;
this CSV supports sample-level CV, holdout testing, and interpretation only.

===== Target threshold: 50 ug/L =====
Full data: n=4038, p=31, class counts={0: 3991, 1: 47}
Dropped before split: []
NoData (-9999) cells converted to missing; counts:
{'Gravitational field': 1, 'TWI': 1, 'CEC': 1, 'Parent material': 1, 'Climate classification': 2, 'Land use': 1, 'G_As_05': 1, 'G_As_10': 1, 'G_As_50': 1, 'pH': 1, 'Precipitation': 2, 'Solar radiatio

Independent test metrics: {'classification_threshold': 0.2, 'TPR': 0.2222222222222222, 'FPR': 0.008760951188986232, 'Accuracy': 0.9826732673267327, 'Precision': 0.2222222222222222, 'Balanced_accuracy': 0.606730635516618, 'MCC': 0.21346127103323598, 'ROC_AUC': 0.6793213739396468, 'PR_AUC': 0.24713305247741862, 'ECE_10_equal_width_bins': 0.006498400950541349}
Model and independent test evaluation saved to E:\NO-NC\normalization\revise\reviewer3_threshold_results\threshold_50
Risk raster and map-level uncertainty require the full predictor rasters;
this CSV supports sample-level CV, holdout testing, and interpretation only.
Consolidated optimal settings and best models saved to E:\NO-NC\normalization\revise\reviewer3_threshold_results
